In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.value import Params, add_value
from src.experiment import n_kontrol, simulasi

sc = pd.read_parquet("../data/scored.parquet"); P = Params(); v = add_value(sc, P)
layak = v[v["ev"] > 0]
base = layak["churned"].mean()
print(len(layak), round(base, 4))      # harapan: 4195 dan 0.4086

4195 0.4086


In [2]:
rows = []
for s in [0.25, 0.20, 0.15, 0.10]:
    for share in [0.5, 0.3, 0.2]:
        nc, nt = n_kontrol(base, s, share)
        rows.append(dict(save_rate=s, porsi_kontrol=share, kontrol=nc, promo=nt, total=nc + nt,
                         muat_di_4195=(nc + nt) <= len(layak),
                         nilai_tertahan_juta=round(nc * layak["ev"].mean() / 1e6, 1)))
t = pd.DataFrame(rows)
print(t.to_string(index=False))
t.to_csv("../reports/experiment_sample_size.csv", index=False)

 save_rate  porsi_kontrol  kontrol  promo  total  muat_di_4195  nilai_tertahan_juta
      0.25            0.5      344    344    688          True                  7.6
      0.25            0.3      246    574    820          True                  5.4
      0.25            0.2      215    860   1075          True                  4.7
      0.20            0.5      545    545   1090          True                 12.0
      0.20            0.3      390    909   1299          True                  8.6
      0.20            0.2      341   1363   1704          True                  7.5
      0.15            0.5      981    981   1962          True                 21.6
      0.15            0.3      701   1635   2336          True                 15.4
      0.15            0.2      614   2453   3067          True                 13.5
      0.10            0.5     2232   2232   4464         False                 49.1
      0.10            0.3     1595   3720   5315         False              

In [3]:
y = layak["churned"].to_numpy()
for s_benar in [0.0, 0.046, 0.10, 0.15, 0.25]:
    h, est = simulasi(y, s_benar)
    print(f"save rate sebenarnya {s_benar:5.1%} | "
          + " | ".join(f"{k} {x:.0%}" for k, x in h.items())
          + f" | perkiraan rata-rata {est:.1%}")

save rate sebenarnya  0.0% | lanjutkan 0% | hentikan 26% | belum jelas 74% | perkiraan rata-rata -0.2%
save rate sebenarnya  4.6% | lanjutkan 2% | hentikan 3% | belum jelas 95% | perkiraan rata-rata 4.4%
save rate sebenarnya 10.0% | lanjutkan 32% | hentikan 0% | belum jelas 68% | perkiraan rata-rata 9.8%
save rate sebenarnya 15.0% | lanjutkan 82% | hentikan 0% | belum jelas 18% | perkiraan rata-rata 14.9%
save rate sebenarnya 25.0% | lanjutkan 100% | hentikan 0% | belum jelas 0% | perkiraan rata-rata 24.9%
